In [ ]:

from google.colab import drive
drive.mount('/content/drive',force_remount=True)
#copy
!cp /content/drive/MyDrive/tomato_dataset_updated-V6.zip /content/

#extract the folder in google colab
!unzip -q -o /content/tomato_dataset_updated-V6.zip  -d /content/dataset

import os
print(os.listdir('/content/dataset/tomato_dataset_updated-V6/archive'))




Mounted at /content/drive


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

IMAGE_SIZE = (260,260)
BATCH_SIZE = 16
TRAIN_DIR = "/content/dataset/tomato_dataset_updated-V6/archive/train"
TEST_DIR = "/content/dataset/tomato_dataset_updated-V6/archive/test"


#we divide the dataset as 80% training , 20% validate
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=False
)
class_names = train_ds.class_names
print(class_names)

#this for prevent the CPU and GPU to be idle while one of them is working

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)

val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)


#create a new file .keras and save the best result in it
checkpoint_path = "/content/drive/MyDrive/AgroSnapMode_ENB2.keras"

# 1. save the best version on result
checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path,
    save_best_only=True,
    monitor='val_accuracy',
    mode='max',
    verbose=1
)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3, #stop training after 3 epochs if the resolution didn't get better
    restore_best_weights=True
)

early_stopping_for_the_whole_model  = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5, #stop training after 3 epochs if the resolution didn't get better
    restore_best_weights=True
)

num_classes = 6 #number of labels/ categories [diseases types]

# Augmentation
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.3),
    layers.RandomZoom(height_factor=(-0.2, 0.2), width_factor=(-0.2, 0.2), fill_mode='reflect'),
    layers.RandomContrast(0.1),
    layers.RandomBrightness(factor=0.1, value_range=(0, 255)),
    ])

# data_augmentation = tf.keras.Sequential([
#     layers.RandomFlip("horizontal_and_vertical"),
#     layers.RandomRotation(0.06),
#     layers.RandomBrightness(0.01)
# ])
# def augment_image(image, label):
#     image = data_augmentation(image)
#     return image, label
# train_ds = train_ds.map(augment_image, num_parallel_calls=tf.data.AUTOTUNE)
# train_ds = train_ds.prefetch(buffer_size=tf.data.AUTOTUNE)

# call the model
base_model = tf.keras.applications.EfficientNetB2(
    input_shape=(260, 260, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False #no need of modificationd of the base weights

# collect all info above and use it to create the model structure
model = models.Sequential([
    data_augmentation,
    base_model,
    layers.GlobalAveragePooling2D(),

    # #1st layer to learn
    # layers.Dense(512),
    # layers.BatchNormalization(),
    # layers.Activation('relu'),
    # layers.Dropout(0.3),


    #2nd layer
    layers.Dense(256),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.4),

    #last layer to take a dicision
    layers.Dense(num_classes, activation='softmax',kernel_regularizer=tf.keras.regularizers.l2(0.001)) #add new layerand this will be detect the desease of plants
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss=tf.keras.losses.CategoricalFocalCrossentropy(gamma=2.0),
    metrics=['accuracy']
)


#we just training the new layer
history_warmup = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[checkpoint_callback, early_stopping]
)



base_model.trainable = True


# we will use it for the layers we need to unlcok it
fine_tune_at = 300

for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False


for layer in base_model.layers[:-100]:
    layer.trainable = False


model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001), #adam is an algorithm
    loss=tf.keras.losses.CategoricalFocalCrossentropy(gamma=2.0),
    metrics=['accuracy']
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=[checkpoint_callback,early_stopping_for_the_whole_model]
)


# =========================================================
# (Confusion Matrix)
# =========================================================



y_true_all = []
y_pred_all = []

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)


    y_true_all.extend(np.argmax(labels.numpy(), axis=1))
    y_pred_all.extend(np.argmax(preds, axis=1))

# Converting lists to Python arrays
y_true_all = np.array(y_true_all)
y_pred_all = np.array(y_pred_all)

# 2. Print the detailed accuracy report for each disease.(Precision & Recall)
print(classification_report(y_true_all, y_pred_all, target_names=class_names))

# 3. Calculating and plotting the confusion matrix
cm = confusion_matrix(y_true_all, y_pred_all)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.ylabel('(Actual)')
plt.xlabel('(Predicted)')
plt.title('Confusion matrix for disease classification results')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


Found 10991 files belonging to 6 classes.
Using 8793 files for training.
Found 10991 files belonging to 6 classes.
Using 2198 files for validation.
Found 2824 files belonging to 6 classes.
['bacterial_spot', 'early_blight', 'healthy', 'late_blight', 'leaf_mold', 'yellow_leaf_curl_virus']
31790344/31790344 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
Epoch 1/10
550/550 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step - accuracy: 0.4474 - loss: 0.2667
Epoch 1: val_accuracy improved from None to 0.79891, saving model to /content/drive/MyDrive/AgroSnapMode_ENB2.keras

Epoch 1: finished saving model to /content/drive/MyDrive/AgroSnapMode_ENB2.keras
550/550 ━━━━━━━━━━━━━━━━━━━━ 100s 141ms/step - accuracy: 0.5614 - loss: 0.2024 - val_accuracy: 0.7989 - val_loss: 0.0870
Epoch 2/10
550/550 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step - accuracy: 0.6889 - loss: 0.1323
Epoch 2: val_accuracy improved from 0.79891 to 0.84031, saving model to /content/drive/MyDrive/AgroSnapMode_ENB2.keras

Epoch 2: finished saving model to /content/dr

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds)
print(f"Accuracy test result is:  {test_accuracy * 100:.2f}%")